# Feature Engineering & Train/Val/Test Split

This notebook consumes the labeled dataset produced by **Labelling Strategy** and:

1. Engineers additional features from raw statistics
2. Inspects feature correlations and per-class distributions
3. Selects the final feature set
4. Performs a **stratified 70/15/15 split** with verified zero group overlap
5. Saves train/validation/test parquet files and a feature schema JSON

## Inputs
- `thermalwatch_labeled.parquet` (output of Labelling Strategy notebook)

## Outputs
| File | Description |
|---|---|
| `data/ml/train.parquet` | Training split (70 %) |
| `data/ml/validation.parquet` | Validation split (15 %) |
| `data/ml/test.parquet` | Test split (15 %) |
| `data/ml/feature_schema.json` | Feature columns, target, exclusions |
| `data/ml/split_summary.json` | Split sizes and class distributions |

In [ ]:
import os

# Paths — adjust if your data directory differs
INPUT_PARQUET = INPUT_PARQUET  # output from Labelling Strategy
OUTPUT_DIR = "data/ml"                          # output: train/val/test parquets

## 1. Imports & Load Data

In [10]:
import pandas as pd
import numpy as np
import json

df = pd.read_parquet(INPUT_PARQUET)
df.shapeimport os


(41812, 20)

In [11]:
df.dtypes

group_id                                 object
obs_count                                 int64
first_seen                  datetime64[us, UTC]
last_seen                   datetime64[us, UTC]
mean_frp                                float64
std_frp                                 float64
latitude                                float64
longitude                               float64
months_active                             int64
monsoon_obs_count                         int64
monsoon_active                             bool
frp_cv                                  float64
nearest_osm_distance_deg                float64
nearest_osm_type                         object
nearest_osm_name                         object
nearest_osm_distance_km                 float64
near_industrial                            bool
label                                    object
label_confidence                        float64
label_reason                             object
dtype: object

In [12]:
df.columns.tolist()

['group_id',
 'obs_count',
 'first_seen',
 'last_seen',
 'mean_frp',
 'std_frp',
 'latitude',
 'longitude',
 'months_active',
 'monsoon_obs_count',
 'monsoon_active',
 'frp_cv',
 'nearest_osm_distance_deg',
 'nearest_osm_type',
 'nearest_osm_name',
 'nearest_osm_distance_km',
 'near_industrial',
 'label',
 'label_confidence',
 'label_reason']

In [14]:
df['label'].value_counts()

,mean_frp,log_mean_frp,std_frp,log_std_frp
count,41812.000000,41812.000000,10578.000000,41812.000000
mean,5.192382,1.512528,1.585760,0.200672
std,16.579523,0.638212,2.893637,0.415994
min,0.000000,0.000000,0.000000,0.000000
25%,1.960000,1.085189,0.631633,0.000000
50%,3.180000,1.430311,1.055277,0.000000
75%,5.430000,1.860975,1.696398,0.081444
max,2160.960000,7.678770,138.111883,4.935279


## 2. Feature Engineering

### 2a. Log-transform FRP features

FRP (Fire Radiative Power) is heavy-tailed. Log1p-transform reduces skew and stabilises variance.

In [14]:
df['log_mean_frp'] = np.log1p(df['mean_frp'])
df['log_std_frp'] = np.log1p(df['std_frp'].fillna(0))

df[['mean_frp', 'log_mean_frp', 'std_frp', 'log_std_frp']].describe()

,mean_frp,log_mean_frp,std_frp,log_std_frp
count,41812.000000,41812.000000,10578.000000,41812.000000
mean,5.192382,1.512528,1.585760,0.200672
std,16.579523,0.638212,2.893637,0.415994
min,0.000000,0.000000,0.000000,0.000000
25%,1.960000,1.085189,0.631633,0.000000
50%,3.180000,1.430311,1.055277,0.000000
75%,5.430000,1.860975,1.696398,0.081444
max,2160.960000,7.678770,138.111883,4.935279


### 2b. Temporal features

- `first_seen_month` — calendar month of first detection (captures seasonal signal)
- `active_duration_days` — span between first and last detection

In [15]:
df['first_seen_month'] = df['first_seen'].dt.month
df['active_duration_days'] = (df['last_seen'] - df['first_seen']).dt.days

df[['first_seen_month', 'active_duration_days']].describe()

,first_seen_month,active_duration_days
count,41812.000000,41812.000000
mean,5.472400,298.312087
std,3.710594,571.925234
min,1.000000,0.000000
25%,2.000000,0.000000
50%,5.000000,0.000000
75%,9.000000,0.000000
max,12.000000,1611.000000


## 3. Feature Correlation & Class Profiling

In [17]:
numeric_cols = ['obs_count', 'mean_frp', 'log_mean_frp', 'std_frp', 'log_std_frp',
                 'months_active', 'monsoon_obs_count', 'frp_cv', 
                 'nearest_osm_distance_km', 'active_duration_days', 'first_seen_month']

df[numeric_cols].corr()

,obs_count,mean_frp,log_mean_frp,std_frp,log_std_frp,months_active,monsoon_obs_count,frp_cv,nearest_osm_distance_km,active_duration_days,first_seen_month
label,,,,,,,,,,,
industrial_thermal_source,58.124392,2.048902,1.069136,1.266566,0.732775,10.518436,6.433549,0.556120,0.518236,1459.602310,2.235008
mining_thermal_source,64.445758,2.316920,1.172360,1.356581,0.818910,10.692281,7.891864,0.564274,0.895304,1297.769471,2.953408
natural_fire,1.056708,6.292375,1.637883,3.078442,0.051716,1.029958,0.340750,0.455578,12.442201,21.300542,6.500000
unknown,1.872500,4.931003,1.528369,1.865056,0.117715,1.356300,0.093900,0.471825,1.170799,102.483000,5.328600


### Per-class mean feature values

Industrial/mining classes show clear separation:
- Much higher `obs_count`, `months_active`, `active_duration_days`
- Much lower `nearest_osm_distance_km`
- Lower `mean_frp` (industrial fires are lower intensity but persistent)

In [16]:
df.groupby('label')[numeric_cols].mean()

,obs_count,mean_frp,log_mean_frp,std_frp,log_std_frp,months_active,monsoon_obs_count,frp_cv,nearest_osm_distance_km,active_duration_days,first_seen_month
obs_count,1.000000,-0.067157,-0.208231,-0.066630,0.585944,0.845657,0.926097,0.111738,-0.279515,0.798434,-0.358693
mean_frp,-0.067157,1.000000,0.485444,0.857282,-0.037255,-0.090685,-0.054329,0.200076,0.069464,-0.089798,0.014715
log_mean_frp,-0.208231,0.485444,1.000000,0.631755,-0.091238,-0.311745,-0.167027,0.264644,0.165236,-0.308575,0.097375
std_frp,-0.066630,0.857282,0.631755,1.000000,0.756302,-0.176794,-0.029500,0.469428,0.156237,-0.155607,0.080614
log_std_frp,0.585944,-0.037255,-0.091238,0.756302,1.000000,0.686400,0.533152,0.738495,-0.243676,0.717953,-0.253229
months_active,0.845657,-0.090685,-0.311745,-0.176794,0.686400,1.000000,0.760190,0.195919,-0.351028,0.948810,-0.388072
monsoon_obs_count,0.926097,-0.054329,-0.167027,-0.029500,0.533152,0.760190,1.000000,0.128034,-0.239266,0.695432,-0.298064
frp_cv,0.111738,0.200076,0.264644,0.469428,0.738495,0.195919,0.128034,1.000000,-0.068122,0.151826,-0.080886
nearest_osm_distance_km,-0.279515,0.069464,0.165236,0.156237,-0.243676,-0.351028,-0.239266,-0.068122,1.000000,-0.342086,0.113334
active_duration_days,0.798434,-0.089798,-0.308575,-0.155607,0.717953,0.948810,0.695432,0.151826,-0.342086,1.000000,-0.403217


## 4. Feature Selection

Final feature set — chosen to minimise redundancy and leakage:

| Feature | Rationale |
|---|---|
| `obs_count` | Strongest single persistence proxy |
| `log_mean_frp` | Log-transformed FRP (replaces `mean_frp`) |
| `log_std_frp` | Log-transformed FRP std (replaces `std_frp`) |
| `frp_cv` | FRP variability ratio |
| `months_active` | Months with detections — very strong separator |
| `nearest_osm_distance_km` | Proximity to industrial features |
| `active_duration_days` | Calendar span |
| `first_seen_month` | Seasonality signal |

**Excluded:**
- `latitude`, `longitude` — excluded to prevent geographic memorization (leakage risk)
- `monsoon_obs_count` — redundant with `months_active` / `obs_count`
- `mean_frp`, `std_frp` — replaced by log-transformed versions

In [ ]:
feature_cols = [
    'obs_count',
    'log_mean_frp',
    'log_std_frp',
    'frp_cv',
    'months_active',
    'nearest_osm_distance_km',
    'active_duration_days',
    'first_seen_month',
]

## 5. Train / Validation / Test Split

**Strategy:** stratified 70 / 15 / 15 split using `sklearn.train_test_split`.

Each row is one unique `group_id` (physical location), so a group-level uniqueness check is performed to guarantee **zero data leakage** across splits.

In [ ]:
# Sanity check: each group_id appears exactly once
assert df['group_id'].nunique() == len(df), "Duplicate group_ids found!"
print("group_id uniqueness verified.")

In [20]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print("Train:", train_df.shape, train_df['label'].value_counts().to_dict())
print("Val:", val_df.shape, val_df['label'].value_counts().to_dict())
print("Test:", test_df.shape, test_df['label'].value_counts().to_dict())

Train: (29268, 24) {'natural_fire': 16800, 'unknown': 7000, 'industrial_thermal_source': 3455, 'mining_thermal_source': 2013}
Val: (6272, 24) {'natural_fire': 3600, 'unknown': 1500, 'industrial_thermal_source': 740, 'mining_thermal_source': 432}
Test: (6272, 24) {'natural_fire': 3600, 'unknown': 1500, 'industrial_thermal_source': 741, 'mining_thermal_source': 431}


In [21]:
assert len(set(train_df['group_id']) & set(val_df['group_id'])) == 0
assert len(set(train_df['group_id']) & set(test_df['group_id'])) == 0
assert len(set(val_df['group_id']) & set(test_df['group_id'])) == 0
print("Verified: zero group overlap across train/val/test.")

Verified: zero group overlap across train/val/test.


## 6. Save Splits & Metadata

In [22]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_df.to_parquet(f"{OUTPUT_DIR}/train.parquet", engine="pyarrow", index=False)
val_df.to_parquet(f"{OUTPUT_DIR}/validation.parquet", engine="pyarrow", index=False)
test_df.to_parquet(f"{OUTPUT_DIR}/test.parquet", engine="pyarrow", index=False)

print("Saved train/validation/test parquet files.")

Saved train/validation/test parquet files.


In [23]:
feature_schema = {
    "feature_columns": feature_cols,
    "target_column": "label",
    "label_classes": sorted(df['label'].unique().tolist()),
    "excluded_features": {
        "latitude": "excluded to avoid geographic memorization (leakage risk)",
        "longitude": "excluded to avoid geographic memorization (leakage risk)",
        "monsoon_obs_count": "redundant with months_active/obs_count",
        "mean_frp": "replaced by log_mean_frp",
        "std_frp": "replaced by log_std_frp"
    },
    "notes": "raw lat/lon deliberately excluded from primary feature set"
}

with open(f"{OUTPUT_DIR}/feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=2)

feature_schema

{'feature_columns': ['obs_count',
  'log_mean_frp',
  'log_std_frp',
  'frp_cv',
  'months_active',
  'nearest_osm_distance_km',
  'active_duration_days',
  'first_seen_month'],
 'target_column': 'label',
 'label_classes': ['industrial_thermal_source',
  'mining_thermal_source',
  'natural_fire',
  'unknown'],
 'excluded_features': {'latitude': 'excluded to avoid geographic memorization (leakage risk)',
  'longitude': 'excluded to avoid geographic memorization (leakage risk)',
  'monsoon_obs_count': 'redundant with months_active/obs_count',
  'mean_frp': 'replaced by log_mean_frp',
  'std_frp': 'replaced by log_std_frp'},
 'notes': 'raw lat/lon deliberately excluded from primary feature set; test separately in notebook 04 to quantify their effect on performance per spec requirement'}

In [24]:
split_summary = {
    "total_rows": int(len(df)),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "split_method": "stratified 70/15/15 split by label; group-level uniqueness verified (one row = one physical source)",
    "train_class_distribution": train_df['label'].value_counts().to_dict(),
    "validation_class_distribution": val_df['label'].value_counts().to_dict(),
    "test_class_distribution": test_df['label'].value_counts().to_dict(),
    "random_state": 42
}

with open(f"{OUTPUT_DIR}/split_summary.json", "w") as f:
    json.dump(split_summary, f, indent=2)

split_summary

{'total_rows': 41812,
 'train_rows': 29268,
 'validation_rows': 6272,
 'test_rows': 6272,
 'split_method': 'stratified 70/15/15 split by label; group-level uniqueness verified (one row = one physical source)',
 'train_class_distribution': {'natural_fire': 16800,
  'unknown': 7000,
  'industrial_thermal_source': 3455,
  'mining_thermal_source': 2013},
 'validation_class_distribution': {'natural_fire': 3600,
  'unknown': 1500,
  'industrial_thermal_source': 740,
  'mining_thermal_source': 432},
 'test_class_distribution': {'natural_fire': 3600,
  'unknown': 1500,
  'industrial_thermal_source': 741,
  'mining_thermal_source': 431},
 'random_state': 42}